In [ ]:
import pandas as pd
train_df = pd.read_parquet(r"final_data\chunk-based-split\train_df_prepared.parquet", engine="pyarrow")
valid_df = pd.read_parquet(r"final_data\chunk-based-split\valid_df_prepared.parquet", engine="pyarrow")
test_df = pd.read_parquet(r"final_data\chunk-based-split\test_df_prepared.parquet", engine="pyarrow")

In [ ]:
cols_to_drop = ["flow_id", "timestamp", "src_ip", "src_port", "dst_ip", "dst_port"]
train_df = train_df.drop(columns=cols_to_drop)
valid_df = valid_df.drop(columns=cols_to_drop)
test_df = test_df.drop(columns=cols_to_drop)

KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score, classification_report

# Tách features và nhãn
X_train, y_train = train_df.drop(columns=["label"]), train_df["label"]
X_valid, y_valid = valid_df.drop(columns=["label"]), valid_df["label"]
X_test, y_test = test_df.drop(columns=["label"]), test_df["label"]

# Các giá trị k (số láng giềng) cần thử nghiệm (ưu tiên số lẻ để tránh tình trạng hòa)
k_values = [3, 5, 7, 9, 11, 15, 21]

best_k = None
best_f1 = -1
best_model = None

print("Đang huấn luyện phân lớp KNN và tìm 'k' tốt nhất dựa trên Validation Macro F1...")

for k in k_values:
    # Sử dụng n_jobs=-1 để sử dụng tối đa CPU cores
    knn = KNeighborsClassifier(n_neighbors=k, n_jobs=-1)
    knn.fit(X_train, y_train)
    
    # Dự đoán trên tập validation
    y_valid_pred = knn.predict(X_valid)
    
    # Tính Macro F1
    current_f1 = f1_score(y_valid, y_valid_pred, average='macro')
    print(f"k = {k:2d} | Validation Macro F1: {current_f1:.4f}")
    
    # Cập nhật model tốt nhất
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_k = k
        best_model = knn

print(f"\n=> K tốt nhất tìm được là k = {best_k} với Validation Macro F1: {best_f1:.4f}")

# Đánh giá trên tập test bằng classification report
print("\n--- Đánh giá Model Tốt nhất (k={}) trên tập TEST ---".format(best_k))
y_test_pred = best_model.predict(X_test)
print(classification_report(y_test, y_test_pred, digits=4))


SVM

In [ ]:
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import f1_score, classification_report
import time

# Với tập dữ liệu rất lớn (~2 triệu mẫu), SVC hoặc LinearSVC truyền thống có thể mất cực kỳ nhiều thời gian/bộ nhớ.
# Sử dụng SGDClassifier với loss='hinge' tương đương với Linear SVM được huấn luyện bằng Stochastic Gradient Descent siêu tốc.

print("Đang huấn luyện phân lớp Linear SVM (bằng SGDClassifier)...")
start_time = time.time()

# Không cần chuẩn hóa StandardScaler do tính chất tập dữ liệu của bạn đã được scale (từ lựa chọn trước)
# Khởi tạo mô hình SGDClassifier cho SVM tuyến tính (loss='hinge'). default: alpha=0.0001
svm_model = SGDClassifier(
    loss='hinge', 
    penalty='l2', 
    alpha=1e-4, 
    max_iter=1000, 
    tol=1e-3, 
    n_jobs=-1,      # Sử dụng tất cả core CPU
    random_state=42 # Đặt seed để mô hình có tính lặp lại
)

# Huấn luyện mô hình
svm_model.fit(X_train, y_train)
train_time = time.time() - start_time
print(f"Huấn luyện hoàn tất trong {train_time:.2f} giây.")

# Dự đoán trên tập validation và tính hàm mục tiêu F1
y_valid_pred = svm_model.predict(X_valid)
valid_f1 = f1_score(y_valid, y_valid_pred, average='macro')
print(f"Validation Macro F1: {valid_f1:.4f}")

# Đánh giá trên tập test bằng classification report
print("\n--- Đánh giá mô hình SVM trên tập TEST ---")
y_test_pred = svm_model.predict(X_test)
print(classification_report(y_test, y_test_pred, digits=4))


Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report
import time

print("Đang huấn luyện phân lớp Random Forest...")
start_time = time.time()

# Cấu hình mặc định tốt cho dữ liệu lớn:
# Bật n_jobs=-1 để song song hoá quá trình dựng từng cây.
# Giới hạn max_depth=20 để tránh tiêu tốn quá mức RAM và tránh Overfitting.
# Cấu hình random_state=42 để model mang tính lặp lại (reproducibility)
rf_model = RandomForestClassifier(
    n_estimators=100, 
    max_depth=20,     # Giới hạn độ sâu phân nhánh
    n_jobs=-1, 
    random_state=42
)

# Huấn luyện mô hình
rf_model.fit(X_train, y_train)
train_time = time.time() - start_time
print(f"Huấn luyện hoàn tất trong {train_time:.2f} giây.")

# Dự đoán trên tập validation và tính điểm
y_valid_pred = rf_model.predict(X_valid)
valid_f1 = f1_score(y_valid, y_valid_pred, average='macro')
print(f"Validation Macro F1 (max_depth=20, n_estimators=100): {valid_f1:.4f}")

# Đánh giá trên tập test bằng classification report
print("\n--- Đánh giá mô hình Random Forest trên tập TEST ---")
y_test_pred = rf_model.predict(X_test)
print(classification_report(y_test, y_test_pred, digits=4))


Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, classification_report
import time

print("Đang huấn luyện phân lớp Decision Tree...")
start_time = time.time()

# Cấu hình Decision Tree: 
# - Dùng criterion='entropy' (Information Gain) theo lựa chọn.
# - Giữ max_depth=20 giống RF để tránh overfitting/quá tải trên 2 triệu mẫu.
dt_model = DecisionTreeClassifier(
    criterion='entropy',
    max_depth=20,
    random_state=42
)

# Huấn luyện mô hình
dt_model.fit(X_train, y_train)
train_time = time.time() - start_time
print(f"Huấn luyện hoàn tất trong {train_time:.2f} giây.")

# Dự đoán trên tập validation và tính Macro F1
y_valid_pred = dt_model.predict(X_valid)
valid_f1 = f1_score(y_valid, y_valid_pred, average='macro')
print(f"Validation Macro F1: {valid_f1:.4f}")

# Đánh giá trên tập test bằng classification report
print("\n--- Đánh giá mô hình Decision Tree trên tập TEST ---")
y_test_pred = dt_model.predict(X_test)
print(classification_report(y_test, y_test_pred, digits=4))


Logistic Regression

In [ ]:
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import f1_score, classification_report
import time

print("Đang huấn luyện phân lớp Logistic Regression...")
start_time = time.time()

# Cấu hình Logistic Regression cho dữ liệu siêu lớn:
# Dùng SGDClassifier với loss='log_loss' hoàn toàn tương đương cấu hình chuẩn toán học với Logistic Regression
# nhưng sử dụng Gradient Descent (SDG) có tốc độ huấn luyện nhanh gắp hàng chục/trăm lần solver 'lbfgs' mặc định.
# Cấu hình penalty='l2' chuẩn hóa phòng tránh overfitting trên 2 triệu features, max_iter=1000 chuẩn ổn định nhất.
logreg_model = SGDClassifier(
    loss='log_loss',  
    penalty='l2',
    alpha=1e-4, 
    max_iter=1000,
    tol=1e-3,
    n_jobs=-1,
    random_state=42
)

# Huấn luyện mô hình
logreg_model.fit(X_train, y_train)
train_time = time.time() - start_time
print(f"Huấn luyện hoàn tất trong {train_time:.2f} giây.")

# Dự đoán trên tập validation và tính Macro F1
y_valid_pred = logreg_model.predict(X_valid)
valid_f1 = f1_score(y_valid, y_valid_pred, average='macro')
print(f"Validation Macro F1: {valid_f1:.4f}")

# Đánh giá trên tập test bằng classification report
print("\n--- Đánh giá mô hình Logistic Regression trên tập TEST ---")
y_test_pred = logreg_model.predict(X_test)
print(classification_report(y_test, y_test_pred, digits=4))
